# Best Model Test-Set Inference

This notebook loads only the selected best saved model and runs inference on the held-out test split. If `outputs/final_xai_analysis/final_xai_analysis_summary.json` exists, the notebook uses that selection. Otherwise it compares the saved best-model metric JSON files and chooses the higher test macro F1.

In [ ]:
# Colab setup
from google.colab import drive
drive.mount('/content/drive')

# Change this path to your project folder in Drive.
PROJECT_DIR = '/content/drive/MyDrive/Explaining-Predictions-of-Machine-Learning-Models'

%cd $PROJECT_DIR
!pip -q install -r requirements.txt

import sys
from pathlib import Path
sys.path.insert(0, str(Path(PROJECT_DIR) / 'src'))

In [ ]:
import json
from pathlib import Path

import h5py
import joblib
import numpy as np
import pandas as pd
from scipy import sparse

from mmimdb.constants import GENRE_LABELS
from mmimdb.data import DatasetPaths, load_labels
from mmimdb.evaluation import multilabel_metrics, threshold_predictions
from mmimdb.final_analysis import _classic_feature_matrix
from mmimdb.models.neural import MMIMDBTorchDataset, make_loader, predict
from mmimdb.splits import load_split_indices
from mmimdb.utils import load_config, resolve_path, save_json
from mmimdb.xai import load_neural_model, patch_classic_classifier_compat

config = load_config('configs/default.yaml')
paths = DatasetPaths.from_config(config)
_, _, test_idx = load_split_indices(resolve_path(config['splits']['output_dir']))
print(f'Test samples: {len(test_idx)}')

In [ ]:
def metric_from_file(path):
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as f:
        payload = json.load(f)
    test = payload.get('test') or payload.get('test_metrics') or {}
    return test.get('macro_f1')

summary_path = Path('outputs/final_xai_analysis/final_xai_analysis_summary.json')
if summary_path.exists():
    with summary_path.open('r', encoding='utf-8') as f:
        summary = json.load(f)
    best_type = summary['best_model']['best_model_type']
    best_path = Path(summary['best_model']['best_model_path'])
else:
    classic_path = Path(config.get('xai', {}).get('classic', {}).get('model_path', 'outputs/models/best/classic_multimodal_best.joblib'))
    neural_path = Path(config.get('xai', {}).get('neural', {}).get('checkpoint_path', 'outputs/models/best/neural_multimodal_best.pt'))
    classic_metric = metric_from_file(Path('outputs/models/best/classic_multimodal_best_metrics.json'))
    neural_metric = metric_from_file(Path('outputs/models/best/neural_multimodal_best_metrics.json'))
    if classic_metric is None and neural_metric is None:
        raise FileNotFoundError('Run scripts/final_xai_analysis.py first or provide best-model metric JSON files.')
    if neural_metric is not None and (classic_metric is None or neural_metric >= classic_metric):
        best_type, best_path = 'neural', neural_path
    else:
        best_type, best_path = 'classic', classic_path

best_path = Path(best_path)
if not best_path.exists():
    best_path = Path(config.get('xai', {}).get(best_type, {}).get('model_path' if best_type == 'classic' else 'checkpoint_path'))
best_path = resolve_path(best_path)
print('Best model type:', best_type)
print('Best model path:', best_path)

In [ ]:
if best_type == 'classic':
    artifact = joblib.load(best_path)
    patch_classic_classifier_compat(artifact['classifier'])
    x_test = _classic_feature_matrix(artifact, paths.hdf5, paths.metadata, test_idx)
    y_true = load_labels(paths.hdf5)[test_idx]
    y_prob = np.asarray(artifact['classifier'].predict_proba(x_test), dtype=np.float32)
    threshold = artifact.get('threshold', 0.5)
else:
    model, cfg, metadata, checkpoint, device = load_neural_model(best_path, paths.metadata)
    dataset = MMIMDBTorchDataset(
        paths.hdf5,
        test_idx,
        vocab_size=int(metadata['vocab_size']),
        max_length=cfg.max_length,
        input_size=cfg.input_size,
    )
    loader = make_loader(dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
    y_true, y_prob = predict(model, loader, device)
    threshold = checkpoint.get('threshold', 0.5)

y_pred = threshold_predictions(y_prob, threshold)
metrics = multilabel_metrics(y_true, y_prob, threshold=threshold)
print(json.dumps({k: metrics[k] for k in ['sample_f1', 'micro_f1', 'macro_f1', 'weighted_f1', 'micro_precision', 'micro_recall', 'hamming_loss']}, indent=2))

In [ ]:
out_dir = Path('outputs/best_model_colab_inference')
out_dir.mkdir(parents=True, exist_ok=True)

rows = []
for row_i, dataset_idx in enumerate(test_idx):
    predicted = [GENRE_LABELS[i] for i in np.flatnonzero(y_pred[row_i])]
    true = [GENRE_LABELS[i] for i in np.flatnonzero(y_true[row_i])]
    top = np.argsort(y_prob[row_i])[::-1][:5]
    rows.append({
        'dataset_index': int(dataset_idx),
        'true_genres': '|'.join(true),
        'predicted_genres': '|'.join(predicted),
        'top5_genres': '|'.join(GENRE_LABELS[i] for i in top),
        'top5_probabilities': '|'.join(f'{float(y_prob[row_i, i]):.6f}' for i in top),
    })

prediction_df = pd.DataFrame(rows)
prediction_df.to_csv(out_dir / 'best_model_test_predictions.csv', index=False)
np.savez_compressed(out_dir / 'best_model_test_predictions.npz', indices=test_idx, y_true=y_true, y_prob=y_prob, y_pred=y_pred)
save_json({'model_type': best_type, 'model_path': str(best_path), 'metrics': metrics}, out_dir / 'best_model_test_metrics.json')

prediction_df.head(10)